# scraper-x, paso a paso

Este cuaderno es un recorrido completo para alguien que nunca ha usado el
scraper: instalar lo que falta, conectar la sesión de X, hacer dos descargas
pequeñas, revisar que lo descargado esté bien y hacer un primer análisis.

Son **20 pasos cortos**. Cada uno hace una sola cosa y dice qué deberías ver.
Ejecuta las celdas en orden, de arriba abajo, con `Shift+Enter`.

En total toma unos 15 minutos, casi todos de espera mientras X responde.

## Qué hace el scraper, en cuatro líneas

Abre **Firefox de verdad**, sin ventana, usando las cookies de un perfil donde
tú ya iniciaste sesión en X. Escribe en el buscador lo mismo que escribirías tú,
baja por los resultados hasta donde X lo deje, y lee cada publicación de la
pantalla: autor, fecha, texto completo, favoritos y enlace. Lo guarda todo en un
archivo CSV.

No usa la API de pago de X. Es, literalmente, un lector automático de la página.

### Las dos formas de pedirle cosas

| Modo | Qué baja | Script |
|---|---|---|
| **Por cuentas** | Todo lo que publicaron ciertas personas, durante años | `descargar_cuentas.py` |
| **Por términos** | Todo lo que se publique con ciertas palabras o hashtags, venga de quien venga | `descargar_busqueda.py` |

### Por qué no es un `for` de diez líneas

X no entrega años de publicaciones de un tirón, y se defiende de quien pregunta
mucho. El scraper está construido alrededor de eso:

- **Parte el trabajo en lotes.** Por cuentas, un lote es *una persona durante un
  trimestre*. Por términos, *un día*. Cada lote que termina deja una marca
  `.done` en disco.
- **Se puede interrumpir.** Si lo paras, o se va la luz, lo vuelves a lanzar y
  sigue por el lote que faltaba. No repite lo ya hecho.
- **Vigila que X no le esconda resultados.** A veces X corta la paginación en
  seco, sin avisar, y devuelve 40 publicaciones donde había 300. El scraper lo
  detecta comparando el ritmo de publicación y vuelve a pedir el tramo que
  falta, partido en dos.
- **Abre los textos largos.** En los resultados de búsqueda, X recorta las
  publicaciones largas a unos 280 caracteres **sin poner puntos suspensivos ni
  ninguna marca**. El scraper hace clic en "Mostrar más" antes de leer.
- **Deduplica por identificador**, no por texto: dos publicaciones idénticas de
  días distintos son dos publicaciones.

### Qué no hace

- No baja retuits sin comentario: el buscador de X no los devuelve.
- No baja publicaciones borradas, ni de cuentas privadas o suspendidas.
- No inicia sesión ni maneja contraseñas. Tú inicias sesión; él reutiliza esa
  sesión.
- No busca por ubicación: X retiró esos operadores (`near:`, `geocode:`) y hoy
  devuelven cero.

---

## Paso 0. Instalar lo que hace falta

Necesitas cuatro cosas en la máquina, y ninguna se instala desde aquí:

| Programa | Enlace |
|---|---|
| Python 3.11 o superior | <https://www.python.org/downloads/> |
| Firefox | <https://www.mozilla.org/firefox/new/> |
| Visual Studio Code (o Jupyter) | <https://code.visualstudio.com/Download> |
| Git, para bajar el proyecto | <https://git-scm.com/download/win> (Windows) · en Mac viene con las herramientas de Apple |

Si estás empezando de cero, hay una guía con capturas de cada paso, pantalla por
pantalla:

- **Windows** → [INSTALACION_WINDOWS.md](INSTALACION_WINDOWS.md)
- **macOS** → [INSTALACION_MAC.md](INSTALACION_MAC.md)

El proyecto se baja así, desde una terminal:

```
git clone https://github.com/manuelcastiblan/scraper-x.git
cd scraper-x
```

**Las librerías de Python sí se instalan desde aquí**, con la celda que sigue.
Es la única celda que puedes saltarte si ya lo hiciste antes.

In [ ]:
# Instala las librerías del scraper y las del análisis.
# Si ya están, no hace nada y lo dice.
%pip install -q -r requirements.txt
%pip install -q pandas matplotlib

print("\nListo. Si aparecieron avisos de 'already satisfied', es que ya estaban.")

## Paso 1. Comprobar que el cuaderno está en la carpeta correcta

El cuaderno llama a los scripts por su nombre, así que tiene que estar **en la
misma carpeta que ellos**. Si abriste el cuaderno desde otro lado, esta celda te
lo dice ahora y no más adelante con un error confuso.

In [ ]:
from pathlib import Path

CARPETA = Path.cwd()
print("Estoy trabajando en:", CARPETA)
print()

necesarios = ["scraper_x.py", "descargar_cuentas.py", "descargar_busqueda.py",
              "requirements.txt", "ejemplo_pacto_historico.json"]
faltan = [n for n in necesarios if not (CARPETA / n).is_file()]

for nombre in necesarios:
    print(("  OK   " if (CARPETA / nombre).is_file() else "  FALTA"), nombre)

assert not faltan, (
    "Abre el cuaderno desde la carpeta scraper-x. Faltan: " + ", ".join(faltan)
)
print("\nTodo en su sitio.")

## Paso 2. Comprobar las versiones

Que Python sea 3.11 o superior y que Selenium esté instalado.

In [ ]:
import sys

sys.path.insert(0, str(CARPETA))
import selenium
import scraper_x          # el scraper, como librería

print("Python  ", sys.version.split()[0])
print("selenium", selenium.__version__)

mayor, menor = sys.version_info[:2]
assert (mayor, menor) >= (3, 11), f"Este proyecto necesita Python 3.11+, y tienes {mayor}.{menor}"
print("\nVersiones correctas.")

---

## Paso 3. Elegir el perfil de Firefox

Aquí está el punto donde más gente se traba, así que vale la pena entenderlo.

El scraper **no sabe tu contraseña y no la quiere**. Lo que hace es abrir
Firefox con un perfil tuyo, y si en ese perfil ya iniciaste sesión en X, las
cookies están ahí y X lo atiende como te atendería a ti.

Un *perfil* es la carpeta donde Firefox guarda tus pestañas, marcadores y
cookies. Puedes tener varios. Lo recomendable es crear uno aparte sólo para el
scraper, para poder seguir navegando mientras descarga:

1. Cierra Firefox del todo.
2. Ábrelo con el administrador de perfiles:
   - **Windows**: `Win+R` → `firefox.exe -P`
   - **macOS**: en la Terminal, `/Applications/Firefox.app/Contents/MacOS/firefox -P`
3. **Create Profile…**, llámalo `scraper`, ábrelo y entra a <https://x.com> con
   tu cuenta.
4. **Cierra Firefox por completo.** Mientras está abierto bloquea el perfil y el
   scraper no puede usarlo.

La celda siguiente lista los perfiles que hay en esta máquina con la fecha de
sus cookies. **El de cookies más recientes suele ser aquel donde acabas de
iniciar sesión.**

In [ ]:
import configparser
import os
import time


def perfiles_disponibles():
    """Lista los perfiles de Firefox y cuándo se tocaron sus cookies."""
    candidatos = [
        Path.home() / "Library/Application Support/Firefox/profiles.ini",   # macOS
        Path.home() / ".mozilla/firefox/profiles.ini",                      # Linux
        Path.home() / "snap/firefox/common/.mozilla/firefox/profiles.ini",  # Linux (snap)
        Path(os.environ.get("APPDATA", "~")) / "Mozilla/Firefox/profiles.ini",   # Windows
        Path.home() / "AppData/Roaming/Mozilla/Firefox/profiles.ini",            # Windows
    ]
    ini = next((ruta for ruta in candidatos if ruta.is_file()), None)
    if ini is None:
        print("No encontré profiles.ini. Tendrás que escribir la ruta a mano en el paso 4.")
        return

    print(ini, "\n")
    configuracion = configparser.ConfigParser()
    configuracion.read(ini, encoding="utf-8")
    for seccion in (s for s in configuracion.sections() if s.startswith("Profile")):
        ruta = Path(configuracion.get(seccion, "Path", fallback=""))
        if configuracion.getboolean(seccion, "IsRelative", fallback=True):
            ruta = ini.parent / ruta
        cookies = ruta / "cookies.sqlite"
        cuando = (
            time.strftime("%Y-%m-%d %H:%M", time.localtime(cookies.stat().st_mtime))
            if cookies.is_file()
            else "sin cookies"
        )
        marca = "   <- marcado como Default" if configuracion.getboolean(seccion, "Default", fallback=False) else ""
        print(f"  {configuracion.get(seccion, 'Name', fallback=seccion):<22} cookies: {cuando}{marca}")
        print(f"    {ruta}")


perfiles_disponibles()

## Paso 4. Declarar el perfil

Copia la ruta del perfil donde iniciaste sesión y pégala abajo, **entre
comillas**. En Windows, cambia las barras `\` por `/`.

> **Si lo dejas en `None`** el scraper toma el perfil marcado como `Default`.
> Funciona si ése es el que tiene la sesión, pero si no, falla en silencio: X
> manda el navegador al formulario de acceso y la descarga termina vacía, sin
> ningún mensaje de error. Por eso conviene declararlo.

`GECKODRIVER` es el programa que conecta Python con Firefox. Déjalo en `None`:
Selenium lo descarga solo la primera vez. Sólo tendrás que tocarlo si tu red
bloquea esa descarga (está explicado en las guías de instalación).

In [ ]:
# ---------------------------------------------------------------------------
PERFIL_FIREFOX = None   # p. ej. "/Users/tu/Library/Application Support/Firefox/Profiles/a1b2c3d4.scraper"
                        # Windows: "C:/Users/TU_USUARIO/AppData/Roaming/Mozilla/Firefox/Profiles/a1b2c3d4.scraper"
GECKODRIVER    = None   # déjalo así salvo que tu red bloquee la descarga
# ---------------------------------------------------------------------------

perfil = scraper_x.discover_firefox_profile(
    Path(PERFIL_FIREFOX).expanduser() if PERFIL_FIREFOX else None
)
driver_path = scraper_x.resolve_driver_path(
    Path(GECKODRIVER).expanduser() if GECKODRIVER else None
)

print("perfil de Firefox:", perfil or "NO ENCONTRADO")
print("  lo elegí porque:", "lo declaraste arriba" if PERFIL_FIREFOX else "es el Default del sistema")
print("geckodriver      :", driver_path or "lo descargará Selenium la primera vez")

assert perfil is not None, "Sin perfil de Firefox no se puede descargar nada."

## Paso 5. Probar la sesión

Esta es la primera celda que **abre Firefox de verdad** (sin ventana visible).
Hace una búsqueda cualquiera y mira qué le responde X. Tarda unos 20 segundos.

Lo que puede salir:

| Resultado | Qué significa | Qué hacer |
|---|---|---|
| `results` | Sesión activa | Seguir al paso 6 |
| `login` | Ese perfil no tiene sesión en X | Volver al paso 3; ojo, puede ser otro perfil |
| `error` | X está bloqueando por exceso de consultas | Esperar 10 minutos y repetir |

> Firefox **no puede estar abierto** con ese mismo perfil. Si lo está, ciérralo
> antes de ejecutar.

In [ ]:
driver = scraper_x.create_driver(perfil, driver_path, True, 25.0)
try:
    driver.get(scraper_x.build_search_url("from:petrogustavo"))
    time.sleep(5)
    estado = scraper_x.page_state(driver)
finally:
    driver.quit()          # cerrar siempre, aunque falle

print("respuesta de X:", estado)
assert estado == "results", (
    "Sin sesión activa en ese perfil. Revisa la lista del paso 3: "
    "quizá iniciaste sesión en otro, o toca volver a entrar a X a mano."
)
print("Sesión lista. Ya puedes descargar.")

---

## Paso 6. Cómo le habla el scraper a X

Antes de descargar, vale la pena ver qué le pide exactamente. El scraper arma
una dirección del buscador avanzado de X, igual que si la escribieras tú en el
navegador.

Los operadores que usa:

| Operador | Significa |
|---|---|
| `from:cuenta` | Sólo publicaciones de esa cuenta |
| `since:` / `until:` | Desde / hasta. `until:` **no incluye** ese instante |
| `"dos palabras"` | La frase exacta, con las comillas |
| `(a OR b)` | Cualquiera de las dos |
| `lang:es` | Sólo en español |

Las fechas se escriben con la hora y el huso completos
(`2026-06-20_00:00:00_UTC`). Sin eso, X interpreta la fecha en **su** huso
horario y te mueve el borde del lote sin avisar.

Esta celda no abre nada ni gasta consultas: sólo arma las direcciones para que
las veas.

In [ ]:
ejemplos = [
    "from:petrogustavo since:2026-06-01_00:00:00_UTC until:2026-07-01_00:00:00_UTC",
    '(#PrioridadNacional OR "Prioridad Nacional")',
    "from:MafeCarrascal -filter:replies",
]

for consulta in ejemplos:
    print("consulta:", consulta)
    print("  ->", scraper_x.build_search_url(consulta))
    print()

---

## Paso 7. Cómo se lanzan los scripts desde el cuaderno

Las descargas son programas de línea de comandos. La función `correr()` los
llama con el mismo Python de este cuaderno —así no importa si tu sistema usa
`python`, `python3` o `py`— y les pasa el perfil que acabas de verificar.

Eso último importa: cada script, por su cuenta, volvería a buscar un perfil y
podría tomar otro distinto del que probaste.

In [ ]:
import subprocess

OPCIONES_SESION = ["--profile", str(perfil)]
if driver_path:
    OPCIONES_SESION += ["--driver", str(driver_path)]


def correr(script, *argumentos, sesion=True, mostrar=4000):
    """Ejecuta un script del proyecto y muestra el final de su salida."""
    orden = [sys.executable, "-u", script, *argumentos]
    if sesion:
        orden += OPCIONES_SESION
    proceso = subprocess.run(orden, capture_output=True, text=True)
    print((proceso.stdout or "")[-mostrar:])
    if proceso.returncode:
        print("--- falló ---")
        print((proceso.stderr or "")[-2000:])
    return proceso.returncode


print("Listo. Cada llamada añadirá:", " ".join(OPCIONES_SESION))

## Paso 8. Ver el plan antes de bajar nada

`--plan-only` responde *"esto es lo que haría"* sin abrir el navegador. Es la
forma barata de confirmar que un censo de cuentas está bien escrito antes de
lanzar una descarga de días.

Usa el censo de ejemplo que viene con el proyecto: los 54 congresistas del Pacto
Histórico del periodo 2022-2026.

In [ ]:
correr("descargar_cuentas.py",
       "--roster", "ejemplo_pacto_historico.json",
       "--window-months", "3",
       "--plan-only",
       sesion=False)      # no abre el navegador, no necesita la sesión

---

## Paso 9. Primera descarga: un día de un hashtag

La descarga más pequeña posible, para ver el flujo completo. Un solo día.

- `--term` es la palabra o el hashtag. Se puede repetir para buscar varias.
- `--end` **no se incluye**: `2026-06-21` baja sólo el 20 de junio.
- `--batch-days 1` significa un lote por día.

Tarda entre uno y dos minutos. Verás líneas que informan de cada lote.

> Si X responde con un bloqueo, el script espera y reintenta solo. Puedes
> interrumpir la celda y volver a ejecutarla: continúa donde quedó.

In [ ]:
correr("descargar_busqueda.py",
       "--term", "#PrioridadNacional",
       "--term", "Prioridad Nacional",
       "--start", "2026-06-20",
       "--end", "2026-06-21",
       "--batch-days", "1",
       "--output", "ejemplo_terminos.csv")

## Paso 10. Mirar lo que bajó

El CSV usa `;` como separador y UTF-8 con BOM, que es lo que Excel espera en
español.

> **Nunca lo abras con `head`, `tail` o `wc -l`.** Dos tercios de los textos
> contienen saltos de línea, porque así se escriben las publicaciones: una fila
> del CSV ocupa varias líneas del archivo y esas herramientas la parten a la
> mitad. Hay que leerlo siempre con un lector de CSV: pandas, el módulo `csv`,
> Excel o R.

In [ ]:
import pandas as pd

df = pd.read_csv("ejemplo_terminos.csv", sep=";", encoding="utf-8-sig")
df["date"] = pd.to_datetime(df["date"], format="ISO8601", utc=True)

print(f"{len(df)} publicaciones, {len(df.columns)} columnas\n")
df[["date", "username", "favorites", "text"]].head(3)

## Paso 11. Qué significa cada columna

| Columna | Qué guarda |
|---|---|
| `username` | La cuenta que publicó, sin la arroba |
| `real_name` | El nombre que muestra el perfil |
| `date` | Fecha y hora en UTC, formato ISO |
| `text` | El texto completo, ya expandido |
| `matched_terms` | Cuáles de tus términos aparecen en el texto |
| `favorites_raw` / `favorites` | Favoritos como los muestra X (`2.1K`) y como número (`2100`) |
| `tweet_id` | Identificador único. Es la clave para deduplicar |
| `url` | Enlace directo a la publicación |
| `scraped_at` | Cuándo se descargó. Útil para auditar |
| `source_query` | La consulta exacta que la trajo |

Las descargas **por cuentas** añaden además `corporation`, `congressperson`,
`district`, `roster_id`, `queried_handle` y `service_intervals`, que vienen del
censo. `queried_handle` es la cuenta que se pidió, y comparada con `username`
sirve para detectar que X devolvió publicaciones de otra persona.

In [ ]:
print("columnas de esta descarga:")
for columna in df.columns:
    print("  ", columna)

## Paso 12. Segunda descarga: una cuenta, un mes

Ahora el otro modo. Una sola cuenta del censo, un mes. Tarda dos o tres minutos.

Fíjate en que las fechas llevan huso horario (`-05:00`, hora de Colombia): el
scraper traduce a UTC por dentro, pero los lotes se cortan en la medianoche
local, que es lo que uno quiere al analizar.

Para tu propio estudio, cambiarías `--roster` por tu censo y quitarías
`--account` para bajar todas las cuentas.

In [ ]:
correr("descargar_cuentas.py",
       "--roster", "ejemplo_pacto_historico.json",
       "--account", "senado-ivan-cepeda",
       "--start", "2026-06-01T00:00:00-05:00",
       "--end", "2026-07-01T00:00:00-05:00",
       "--window-months", "1",
       "--retries", "6", "--retry-backoff", "300",
       "--output", "ejemplo_cuenta.csv")

## Paso 13. Cómo hace para poder continuar

Mira qué quedó en disco. Cada lote terminado es un CSV con su marca `.done` al
lado. Cuando relanzas la descarga, el script ve esas marcas y se salta esos
lotes.

Por eso una descarga de días se puede interrumpir sin perder nada, y por eso
puedes apagar el computador a mitad de camino.

In [ ]:
for carpeta in sorted(CARPETA.glob("ejemplo_*_partes")):
    print(carpeta.name + "/")
    for archivo in sorted(carpeta.rglob("*"))[:8]:
        print("   ", archivo.relative_to(carpeta))
    print()

print("Un .done al lado de un .csv significa: ese lote ya está completo.")
print("Por cuentas hay una subcarpeta por persona; por términos, un archivo por día.")

---

## Paso 14. Revisar la calidad: lo básico

Antes de analizar, tres comprobaciones. La primera es la obvia: que no haya
duplicados ni textos vacíos.

In [ ]:
datos = pd.read_csv("ejemplo_cuenta.csv", sep=";", encoding="utf-8-sig")
datos["date"] = pd.to_datetime(datos["date"], format="ISO8601", utc=True)

print("publicaciones   :", len(datos))
print("ids duplicados  :", int(datos["tweet_id"].duplicated().sum()))
print("textos vacíos   :", int(datos["text"].str.strip().eq("").sum()))

if "queried_handle" in datos:
    distinto = (datos["username"].str.lower() != datos["queried_handle"].str.lower()).sum()
    print("autor != cuenta :", int(distinto), " (debe ser 0: si no, X devolvió a otra persona)")

## Paso 15. Revisar la calidad: los textos cortados

Ésta es la comprobación menos evidente y la más importante.

X recorta las publicaciones largas a unos 280 caracteres **sin dejar ninguna
marca**: no hay puntos suspensivos, la frase simplemente se interrumpe. Si el
scraper no hubiera hecho clic en "Mostrar más", tendrías un corpus que parece
correcto y está mutilado.

Se detecta por la forma de las longitudes. Si muchísimas publicaciones miden
entre 270 y 285 caracteres y casi ninguna pasa de 300, no es que la gente
escriba así: es que están cortadas justo ahí.

In [ ]:
largo = datos["text"].str.len()
banda = int(largo.between(270, 285).sum())
largas = int((largo > 300).sum())

print("longitud media  : %.0f caracteres" % largo.mean())
print("la más larga    : %d" % largo.max())
print("entre 270 y 285 : %d  (%.1f%% del total)" % (banda, 100 * banda / len(datos)))
print("más de 300      : %d" % largas)
print()

# La señal no es el tamaño de la banda por sí solo, sino que la banda acumule
# muchas más publicaciones que el tramo de textos largos: si de verdad hubiera
# publicaciones extensas, estarían por encima de 300 y no apiladas en 280.
if len(datos) < 300:
    print("Muestra pequeña: esta comprobación necesita unos cientos de")
    print("publicaciones para ser fiable. Con una cuenta y un mes, tómala como ilustración.")
elif 100 * banda / len(datos) > 20 and banda > 3 * largas:
    print("SOSPECHA de textos cortados. Vuelve a pasar esos lotes con --force --refresh")
else:
    print("Sin señales de truncamiento sistemático.")

## Paso 16. Revisar la calidad: los huecos

Un silencio largo puede ser real —un receso legislativo, una cuenta inactiva— o
ser un tramo que X nunca entregó. No hay forma automática de distinguirlos, pero
sí de señalar los sospechosos: se compara cada hueco con el ritmo de publicación
de esa misma cuenta.

Un mes sin publicar en alguien que publica tres veces al día merece una mirada;
en alguien que publica una vez al mes, no.

In [ ]:
def huecos(tabla, dias_minimos=14, esperadas_minimas=20):
    """Lista los silencios largos de cada cuenta según su propio ritmo."""
    filas = []
    for cuenta, grupo in tabla.groupby("queried_handle"):
        fechas = grupo["date"].dt.tz_convert("UTC").dt.date.sort_values().tolist()
        if len(fechas) < 2:
            continue
        ritmo = len(fechas) / max((fechas[-1] - fechas[0]).days, 1)
        for anterior, siguiente in zip(fechas, fechas[1:]):
            dias = (siguiente - anterior).days
            if dias >= dias_minimos and dias * ritmo >= esperadas_minimas:
                filas.append({"cuenta": cuenta, "desde": anterior, "hasta": siguiente,
                              "días": dias, "esperadas": round(dias * ritmo)})
    return pd.DataFrame(filas).sort_values("esperadas", ascending=False) if filas else pd.DataFrame()


sospechosos = huecos(datos)
print("huecos que merecen revisión:", len(sospechosos))
sospechosos.head()

---

## Paso 17. Primer análisis

Nada sofisticado: volumen por día y las publicaciones con más favoritos. Lo
justo para confirmar que el corpus tiene sentido.

In [ ]:
por_dia = datos.set_index("date").resample("D").size()

print("publicaciones por día:")
print(por_dia.to_string())

In [ ]:
print("las de más favoritos:\n")
columnas = ["date", "username", "favorites", "text"]
print(datos.nlargest(3, "favorites")[columnas].to_string(index=False, max_colwidth=70))

## Paso 18. Un gráfico

Si instalaste matplotlib en el paso 0, esta celda dibuja el volumen diario.

In [ ]:
try:
    import matplotlib.pyplot as plt

    eje = por_dia.plot(kind="bar", figsize=(10, 3), title="Publicaciones por día")
    eje.set_xlabel("")
    eje.set_ylabel("publicaciones")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Instala matplotlib para ver el gráfico:  %pip install matplotlib")

## Paso 19. Qué esperar cuando la descarga sea de verdad

Lo que acabas de hacer son dos lotes. Un corpus real son miles, y X pone
límites:

- Aguanta unas **12 búsquedas nuevas** seguidas y después bloquea entre 9 y 15
  minutos. Cuanto más larga la sesión, más duro se pone.
- El corpus de ejemplo —54 cuentas, cuatro años, 212.962 publicaciones— tomó
  **varios días** de descarga con interrupciones.

Por eso las descargas largas no se lanzan desde el cuaderno, que dejaría el
kernel ocupado días, sino desde una terminal:

```
nohup caffeinate -i python -u descargar_cuentas.py --roster censo.json \
    --profile "$X_FIREFOX_PROFILE" \
    --window-months 3 --retries 12 --retry-backoff 600 >> salida.log 2>&1 &
```

En Windows, con `Start-Process` (está en la guía de instalación). Y mientras
corre puedes volver a este cuaderno y leer el CSV: se reescribe completo después
de cada lote.

## Paso 20. Pasar a tu propio estudio

1. **Arma tu censo.** Copia `censo_plantilla.json` a `censo.json` y llénalo.
   Verifica cada usuario de X contra una fuente institucional: un usuario
   equivocado **no da error**, devuelve cero publicaciones y te deja un hueco
   que parece silencio. En el corpus de ejemplo hubo dos cuentas mal anotadas, y
   sólo se descubrieron al revisar quién aparecía mencionado.
2. **Lanza la descarga** desde la terminal, como en el paso 19.
3. **Al terminar**, pasa `reparar_truncamientos.py` y repite los pasos 14 a 16
   sobre el corpus completo. Ahí sí las comprobaciones son fiables, porque hay
   volumen.
4. **Antes de publicar el corpus**, lee la sección 12 del `README.md`: hay
   consideraciones éticas que no son opcionales.

El `README.md` explica el resto: el formato del censo, los parámetros de cada
script y qué hacer cuando X se pone difícil.